# L15a: Testing a Trading Strategy

Today, we'll choose one strategy and test it twice. We'll backtest it on the real market and then stress-test it on synthetic markets. We'll use betting against beta (BAB) to weight the portfolio in both tests.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe the trading engine.__ Explain how the strategy choices and the six triggers govern a new decision.
> * __Backtest a strategy.__ Run a strategy on the real market and compare its wealth with `SPY`.
> * __Stress-test a strategy.__ Read the range of outcomes over many synthetic markets and apply the five gates.

> __What you need to know:__
>
> __Core concepts:__
>
> * The engine values the portfolio at each close. A trigger starts a new decision, and the orders fill at the next open.
> * A backtest follows one historical market. Each decision uses data through that day's close.
> * A stress test runs the same strategy on many synthetic markets. The five gates judge typical outcomes and the frequency of losses.
>
> __For understanding:__
>
> * How the daily update gives more weight to recent prices.
> * How the preference score and the optional hedge use market sensitivity.
> * The settings of the six triggers, such as the 8% drawdown.

__Notation.__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L15a) lists the symbols with their meanings and units.

___

## Examples

__We will work through these examples in lecture.__

> [▶ Backtest a strategy from the menu](CHEME-5660-L15a-Example-Strategy-Backtest-Fall-2026.ipynb). Inspect a BAB portfolio and trace its wealth on the real market.
>
> [▶ Stress-test a strategy with the five gates](CHEME-5660-L15a-Example-Strategy-Stress-Test-Fall-2026.ipynb). Run the same strategy on 200 synthetic markets and read its scorecard.

__Review on your own.__

> [▶ Update the single index model and the preference weights](CHEME-5660-L15a-Example-Updating-SIM-Gamma-Fall-2026.ipynb). See how daily updates change a stock's market sensitivity and the basket we pick.

___

## The Trading Engine

A strategy makes four choices. The engine follows the strategy one trading day at a time.

| Choice | What it sets |
|:--|:--|
| Picking rule | The tickers in the basket. |
| Weighting rule | How much of the portfolio to hold long or short in each ticker. |
| Market hedge | Whether to apply a cap on the portfolio's market sensitivity. The cap falls as the market falls. |
| Estimates | Keep the 2016 to 2024 fit fixed, or update it at each close with more weight on recent days. |

We'll choose the highest preference scores in each sector. The score ranks stocks using expected growth and their sensitivity to the current market. The [strategy menu](strategies/CHEME-5660-L15a-Strategy-Menu-Fall-2026.ipynb) lists the other choices.

__Betting against beta.__ The BAB weighting rule ranks the picked stocks by their market sensitivity, or beta. It buys the lower-beta stocks and shorts the higher-beta stocks. The target sizes aim to offset their market exposure, and the trading limits can leave the resulting portfolio with a beta above or below zero.

The engine fills the previous decision's orders at the open. It values the portfolio and checks for a new decision at the close.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Backtest-Day.svg" width="860" style="max-width:100%; height:auto;" alt="One time axis with three trading days, day t minus 1, day t, and day t plus 1, each drawn as a bar from its open to its close. A dashed gray arc from the close of day t minus 1 to the open of day t reads: fill yesterday's orders. An arrow rises from the close of day t to a box labeled after the close: compute the wealth W of t, update the estimates, and check the six triggers. A gray arrow to the left of the box reads: no trigger, keep the portfolio. An orange arrow labeled a trigger fires rises to an orange box, make a new decision, with three steps. Step 1 picks a basket with the picking rule. Step 2 weights it with the weighting rule within the trading limits, and within the market sensitivity limit if the hedge is on. Step 3 queues the orders. An arrow from step 3 runs down to the open of day t plus 1, labeled orders fill at the open."/>
    </center>
</div>

### Daily Backtest Algorithm

__Initialize.__ Choose a strategy and set the starting wealth $W_{0}$ in cash. Begin with estimates from the training data.

For each trading day $t$ __do__:

1. __Fill.__ Fill the queued orders at today's open and charge trading costs.
2. __Value and update.__ Accrue interest on cash, value the holdings at the close, and update the single index estimates if chosen.
3. __Decide.__ If this is the first close or a trigger fires, __then__ pick a basket and weight it within the trading limits. Queue the orders for the next open and forecast wealth over the next 21 trading days. Otherwise, keep the current holdings.

__Output.__ Record daily wealth, decisions and their triggers, and trades.

### The Six Triggers

The market state $\xi_{t}$ compares a short and a long moving average of the `SPY` price. It turns positive when the short average falls below the long one. With the hedge on, a `SPY` short can start when $\xi_{t}$ rises above one.

| Trigger | Fires when |
|:--|:--|
| Scheduled | 21 trading days have passed since the last decision. |
| Market turn | The market state changes sign since the last decision. With the hedge on, crossing one also counts. |
| Band exit | Wealth leaves the middle 95% of its forecast. |
| Drawdown | Wealth falls more than 8% below its peak. The peak resets at the drawdown decision. |
| Market sensitivity over its limit | With the hedge on, the portfolio's beta exceeds its limit by more than 0.1. |
| Back from cash | 21 trading days have passed since the last decision and that decision put the portfolio in cash. |

___

## Backtesting a Strategy

We run the engine on the historical prices from January 2, 2025, to November 24, 2026. Our strategy uses gamma by sector to pick the basket and BAB to weight it. We use fixed estimates and leave the optional market hedge off.

Let's look at the weights from one decision. Which stocks are long, and which are short?

> __Example:__
>
> [▶ Backtest a strategy from the menu](CHEME-5660-L15a-Example-Strategy-Backtest-Fall-2026.ipynb). In Task 1, read the long and short weights. Then compare the strategy's wealth with `SPY` and find a day a trigger fired.

The backtest gives us one wealth path. We use the same strategy to see how different markets could change that outcome.

___

## Stress-Testing a Strategy

Each synthetic market shares the real history through November 24, 2026. We append 252 trading days drawn from the general factor model. The strategy starts from the same wealth and estimates on all 200 markets.

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Stress-Test.svg" width="860" style="max-width:100%; height:auto;" alt="Thin red lines fan out from one point on a vertical dashed line labeled start day. The point is the starting wealth W 0. Each line is the wealth path on one synthetic market from the generator, over one synthetic year of 252 trading days, and the fan widens over the year. At the right edge, a sideways histogram shows the wealth on the last day over the 200 synthetic markets. A dashed horizontal line marks the starting wealth W 0, and a solid tick above it marks the median. The bars below W 0 are a darker red, labeled failures: end below W 0. The lines that end below W 0 have the same darker red, and the other lines and bars are a light red."/>
    </center>
</div>

A 21-day wealth forecast helps the engine decide when to trade. The one-year stress test shows how the strategy fares across markets.

### Stress-Test Algorithm

__Initialize.__ Choose a strategy, the real history through the start day, and a generator for $P=200$ synthetic markets.

For each market $p=1,2,\ldots,P$ __do__:

1. __Draw.__ Simulate daily growth rates for one year. Compound the prices from their last real closes.
2. __Run.__ Make the first decision after the last real close and run the daily backtest loop through the synthetic year.
3. __Score.__ Record the Sharpe ratio, largest drawdown, ending wealth, and net present value.

__Output.__ Use the scores across all markets to evaluate the five gates.

### The Five Gates

| Gate | Passes when |
|:--|:--|
| Median Sharpe ratio | At least 0.3. |
| Median largest drawdown | At most 25%. |
| Failure rate | At most 10% of markets end below the starting wealth $W_0$. |
| Median ending wealth | At least the comparison's median ending wealth. |
| Median net present value | At least USD 0 after discounting ending wealth at the Treasury bill rate and subtracting $W_0$. |

For the fourth gate, a strategy with fixed estimates faces a benchmark of 60% `SPY` and 40% `TLT`, a long-term Treasury bond fund. A strategy with updated estimates faces the same strategy with fixed estimates on the same markets.

Let's see how BAB fares over the synthetic year.

> __Example:__
>
> [▶ Stress-test a strategy with the five gates](CHEME-5660-L15a-Example-Strategy-Stress-Test-Fall-2026.ipynb). In Tasks 1 and 2, read the fan of wealth paths and the five-gate scorecard for the strategy we backtested.

In the practicum, the engine fills orders at the decision-day close, which can change how the same strategy scores.

___

## Optional Advanced Material

> [▶ Read the trading engine details](advanced/CHEME-5660-L15a-Reference-Trading-Engine-Fall-2026.ipynb). See how the preference score uses the estimates and how the market hedge sets a beta cap.
>
> [▶ Derive the daily update of the single index model](advanced/ewls/CHEME-5660-L15a-Derivation-EWLS-Update-Fall-2026.ipynb). Derive the running sums and see how the 2016 to 2024 fit sets their starting values.

___

## Summary

> __Key Takeaways:__
>
> * __One engine runs the strategy.__ At each trigger the engine picks and weights a basket. With BAB, lower-beta stocks go long and higher-beta stocks go short.
> * __A backtest gives one historical outcome.__ We compared the strategy's wealth with `SPY` and read the triggers on the wealth path.
> * __A stress test gives a range of outcomes.__ We used four medians and the fraction of markets that ended below starting wealth to judge the same strategy.

A paper test runs the strategy in a paper account at live market prices.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___